# Notebook 3 — Review a SQL answer before sending it

Run a greeting, an unsupported question, a write request, and a card lookup. Then open the SQL and result for the Pisek count and practice editing the draft. The question cells use local Qwen.


In [1]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
if not (HERE / "text2sql.py").exists():
    raise SystemExit("Set the notebook's working directory to 17_agentic_text2sql.")
sys.path.insert(0, str(HERE))

import text2sql as desk
from text2sql import ask

assistant = desk.get_assistant()
assistant.get_graph().print_ascii()


                                                                                  +-----------+......                                                                                 
                                                                                  | __start__ |....................                                                                   
                                                                                 .+-----------+      ............  ..............                                                     
                                                                             ....                                .............   .............                                        
                                                                          ...                                                 .............   ..............                          
                                                                        ..           

## How to read the picture

```text
route            greeting                  -> chat -> END
                 delete / update / insert  -> refuse -> END
                 anything else             -> link

after link       no table                  -> abstain
                 tables found              -> generate -> guard -> execute

after execute    error, or a suspect zero, -> repair once, then guard
                 and a repair is left
                 still broken, or no rows  -> abstain
                 rows                      -> draft

after draft      count, sum, average, or   -> human_review
                 any numeric cell
                 a label, such as a card   -> END
                 type

after human      approve or edit           -> END
                 reject, repair left       -> repair
                 reject, repair spent      -> abstain
```

`human_review` calls `interrupt`. LangGraph runs that function again from the top when the person resumes, and `interrupt` then returns the decision. Do not put work above that call. It would run twice.

Edit replaces the sentence. There is no second model call. The person is the authority.


## 1. Run four questions and compare their paths

- A greeting goes straight to `chat`.
- The question about HarborDesk leave returns `NOT_IN_DATABASE`.
- A request to delete loans stops at `refuse` before the model writes SQL.
- Client 13539's card type comes back as `junior`. The graph sends a label directly.

In the next cell, look for the extra review step when the result is a count.


In [2]:
def show(question):
    values, payload = ask(question)
    print("Q:", question)
    for step in values["steps"]:
        print(" ", step)
    if payload:
        print("PAUSED:", payload["reason"])
        print(payload["sql"])
        print(payload["draft"])
    else:
        print(values["answer"])
    print()
    return values, payload

show("Good morning")
show("How many annual leave days does a full-time HarborDesk employee receive?")
show("Delete every loan whose status is D.")
card, card_pause = show("What type of credit card does the client with id 13539 own?")
print("card waited for a person:", card_pause is not None)


Q: Good morning
  chat (database stays closed)
Good morning! How can I help?

Q: How many annual leave days does a full-time HarborDesk employee receive?
  link -> (none)
  abstain
NOT_IN_DATABASE: the Berka bank tables do not answer that question.

Q: Delete every loan whose status is D.
  refuse write
REFUSED: this desk can only read the database. It will not insert, update, or delete.



Q: What type of credit card does the client with id 13539 own?
  link -> card, client, disp
  generate SELECT c.type FROM card AS C JOIN disp AS D ON C.disp_id = D.disp_id WHERE D.client_id = 13539 AND D.type = 'OWNER'
  guard ok
  execute 1 row(s)
  sentence
The client with id 13539 owns a junior credit card.

card waited for a person: False


## 2. Verify the Pisek count

Open the SQL and check that it filters `A2 = 'Pisek'`. The result grid should show **46 accounts**. The graph pauses on this result so you can check the count before sending the answer.


In [3]:
pisek, pause = ask("How many accounts are there in the district of Pisek?")
for step in pisek["steps"]:
    print(" ", step)
print()
print(pause["reason"])
print(pause["sql"])
print(pause["draft"])
print("rows:", pause["rows"])
print("thread:", pisek["thread_id"])


  link -> account, district
  generate SELECT COUNT(*) FROM account JOIN district ON account.district_id = district.district_id WHERE A2 = 'Pisek'
  guard ok
  execute 1 row(s)
  sentence

The result is a measured number. A person signs the sentence before it is sent to the client.
SELECT COUNT(*) FROM account JOIN district ON account.district_id = district.district_id WHERE A2 = 'Pisek'
There are 46 accounts in the district of Pisek.
rows: [[46]]
thread: branch-fb67292b


## 3. Resume with your edit

Use the same thread ID and choose `edit`. The text you enter becomes the answer.

You can also try `reject` in a fresh run. The graph can write another query if it has a repair left. If it returns another draft and you reject again, the graph abstains.


In [4]:
final, again = ask(
    "",
    thread_id=pisek["thread_id"],
    resume={"action": "edit", "text": "CHECKED: Pisek has 46 accounts."},
)
print("paused again:", again is not None)
print("human_action:", final["human_action"])
print(final["answer"])


paused again: False
human_action: edit
CHECKED: Pisek has 46 accounts.


## Try it in Streamlit

```bash
streamlit run app.py
```

| Button | What it does |
|---|---|
| Approve and send | Sends the current draft |
| Send the edited text | Sends your edited sentence |
| Reject and try another SQL | Tries another query if a repair is available |

The form submits your edit and decision together. The checkpointer keeps the pause in memory, so restart the question if you restart the app.
